# Two-Stage Retrieval with LLM Re-Ranking

First-stage vector search prioritizes retrieval speed and recall, often returning noisy or tangential passages. Two-stage retrieval pairs an initial high-recall FAISS retrieval (top 10-20 candidates) with an LLM-as-a-reranker (Groq `openai/gpt-oss-120b`) that assesses deep relevance and reorders the most pertinent chunks for generation.

## Workflow Architecture

<div align="center">
  <img src="workflow_reranking.png" alt="Two-Stage Retrieval with LLM Re-Ranking Architecture Diagram" width="580" />
</div>

<details>
<summary><b>Click to expand Colorful Mermaid Source Code</b></summary>

```mermaid
flowchart TD
    subgraph In["User Query"]
        Q(["1. User Query"]):::startNode
    end
    subgraph Stage1["Stage 1: Fast Dense Retrieval (High Recall)"]
        DenseRet["2. Vector Search (FAISS + MiniLM)<br/>(Retrieves Top-K: e.g. 20 Chunks)"]:::stage1Node
        BroadChunks["Broad Candidate Pool (20 Chunks)"]:::poolNode
    end
    subgraph Stage2["Stage 2: Precision Re-Ranking (High Precision)"]
        ReRanker["3. LLM / Cross-Encoder Re-Ranker<br/><b>Groq gpt-oss-120b</b><br/>(Scores Query-Document Relevance Pairs)"]:::stage2Node
        TopK["Top-N Re-Ranked Chunks (Top 3 Chunks)"]:::topNode
    end
    subgraph Gen["Stage 3: Generation"]
        Synthesizer["4. Final LLM Synthesis<br/>(Synthesizes with Top Precision Chunks)"]:::synthNode
        Ans(["5. Highly Accurate Grounded Answer"]):::endNode
    end
    Q --> DenseRet
    DenseRet --> BroadChunks
    BroadChunks --> ReRanker
    Q --> ReRanker
    ReRanker --> TopK
    TopK --> Synthesizer
    Q --> Synthesizer
    Synthesizer --> Ans
    classDef startNode fill:#E8F5E9,stroke:#2E7D32,stroke-width:2px,color:#1B5E20;
    classDef stage1Node fill:#E0F7FA,stroke:#00838F,stroke-width:2px,color:#004D40;
    classDef poolNode fill:#FFF8E1,stroke:#FFA000,stroke-width:2px,color:#E65100;
    classDef stage2Node fill:#EDE7F6,stroke:#5E35B1,stroke-width:2px,color:#311B92;
    classDef topNode fill:#E8EAF6,stroke:#3F51B5,stroke-width:2px,color:#1A237E;
    classDef synthNode fill:#E3F2FD,stroke:#1565C0,stroke-width:2px,color:#0D47A1;
    classDef endNode fill:#FFEBEE,stroke:#D32F2F,stroke-width:2px,color:#B71C1C;
```
</details>

### Key Retrieval Principles
- **Two-Stage Pipeline**: Balances high recall in Stage 1 with high precision in Stage 2.
- **Deep Semantic Scoring**: Groq LLM evaluates fine-grained relevance beyond vector similarity.
- **Noise Reduction**: Filters out irrelevant context before final answer generation.


In [21]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("langchain_sample.txt")
raw_doc = loader.load()

raw_doc

[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.\nLangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.\nRetrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.\nBM25 is a traditional sparse retrieval method that scores documents based on keyword matching. Although fast, it often struggles with synonyms and semantic similari

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap=50)
docs = splitter.split_documents(raw_doc)
docs

[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.\nBM25 is a traditional 

In [23]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
vectorStore = FAISS.from_documents(docs , model)
retriver = vectorStore.as_retriever(search_kwargs={"k":8})
retriver

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9304.01it/s]


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000265B13BED50>, search_kwargs={'k': 8})

In [24]:
query = "How can i use langchain to build an application with memory and tools?"
retrieved_docs = retriver.invoke(query)

doc_lines = [f"{i}. {doc.page_content}" for i, doc in enumerate(retrieved_docs)]

doc_lines

['0. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.',
 '1. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.',
 '2. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.',
 '3. FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and comp

In [25]:
from langchain_community import retrievers
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from
most to least relevant to the user's question.
User Question: "{question}"
Documents:
{documents}
Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant.
Output format: comma-separated document indices (e.g., 2,1,3,0,...)
""")


# Use the most stable, globally available model
llm = init_chat_model(model="groq:openai/gpt-oss-120b")

chain = prompt | llm | StrOutputParser()

formatted_docs = "\n".join(doc_lines) #  joining a list of strings into one big string
print(formatted_docs)
response = chain.invoke({"question": query, "documents": formatted_docs})
print()
print(response)

0. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.
Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.
1. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.
2. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.
3. FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and compressed indexes,

In [26]:
indices = [int(i.strip()) for i in response.strip().split(",") if i.strip().isdigit()]
print(indices)
reranked_docs = [retrieved_docs[i] for i in indices if 0 <= i < len(retrieved_docs)]
reranked_docs

[0, 1, 3, 2, 4, 5]


[Document(id='fa32f5b1-b7ea-47fd-b77b-a5259d9f0def', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='cec10f85-04a8-4c07-acd6-ff6ca7624afa', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='2e996d25-6d40-447c-a8f5-c1426aed045f', metadata={'source': 'langchain_sample.txt'}, page_content='FAISS is a popular library used for fast approximate nearest neighbor search 